# Pipeline de classificação binária com validação cruzada, atenção espacial e interpretabilidade

Pipeline de ponta a ponta que gera um dataset sintético, compara modelos clássicos com duas redes neurais e explica as previsões da CNN com Gradient SHAP.

## O que o projeto faz

1. **Dados sintéticos.** Gera 30.000 amostras com 64 features ("pixels" de uma imagem 8x8). O rótulo depende só de `pixel_0`, `pixel_15`, `pixel_32` e `pixel_45`, mais ruído. Cerca de 15% das amostras são positivas.
2. **Engenharia de features com Polars.** Cria `pixel_energy_sum` e `interaction_core` e atribui folds estratificados por classe.
3. **Validação cruzada de 10 folds** para KNN e Árvore de Decisão (acurácia média e desvio).
4. **Split sem vazamento.** Fold 0 é teste, fold 1 é validação (early stopping) e os 8 restantes são treino.
5. **Modelos comparados no fold de teste:**
   - KNN (hiperparâmetros por GridSearch)
   - Árvore de Decisão (critério entropia, poda por `ccp_alpha`)
   - **CNN com atenção espacial:** bloco residual com self-attention sobre o grid 8x8
   - **QNN:** MLP com ativações seno e cosseno em espaço de 32 dimensões. É uma simulação clássica, não roda circuito quântico.
6. **Treino das redes:** AdamW, cosine annealing, `BCEWithLogitsLoss` e early stopping independente por modelo, com restauração dos melhores pesos.
7. **Métricas:** acurácia, F1, ROC AUC e log loss. Como a classe positiva é minoritária, compare por F1, AUC e log loss, não pela acurácia.
8. **Interpretabilidade:** Gradient SHAP (expected gradients) na CNN. Os pixels dominantes devem ser os quatro que geram o rótulo, o que serve de checagem de sanidade.
9. **Saídas** na pasta `out/`:
   - `pipeline_output.parquet`: fold de teste com as previsões dos 4 modelos
   - `curvas.html`: curvas de treino e validação (Altair)
   - `qnn_norms.html`: evolução da norma dos parâmetros da QNN (Plotly)
   - `densidades.png`: distribuição das probabilidades por modelo e classe (plotnine)
   - `shap_top10.png`: top 10 pixels por importância SHAP (seaborn objects)

## Dependências

```
pip install polars pyarrow torch scikit-learn altair plotly plotnine seaborn matplotlib
```

No Colab, `!pip install -q polars pyarrow plotnine` basta. Ative a GPU para acelerar o treino das redes.

## Resultados de referência (fold de teste, n = 3001)

| Modelo | F1 | ROC AUC | Log loss |
|---|---|---|---|
| KNN | 0.10 | 0.877 | 0.310 |
| Decision Tree | 0.90 | 0.984 | 0.132 |
| CNN + Attention | 0.91 | 0.995 | 0.070 |
| QNN | 0.92 | 0.997 | 0.054 |

Os valores variam um pouco entre execuções e entre CPU e GPU.

In [2]:
!pip install -q polars pyarrow plotnine

In [3]:
import copy
from pathlib import Path
import altair as alt
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import plotly.express as px
import polars as pl
import seaborn.objects as so
import torch
import torch.nn as nn
import torch.optim as optim
from plotnine import (
    aes,
    element_text,
    facet_wrap,
    geom_density,
    ggplot,
    labs,
    theme,
    theme_minimal,
)
from sklearn.metrics import accuracy_score, f1_score, log_loss, roc_auc_score
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
N_SAMPLES = 30_000
N_FEATURES = 64
N_SPLITS = 10
TEST_FOLD, VAL_FOLD = 0, 1
N_EPOCHS = 30
PATIENCE = 4
OUT = Path("out")
OUT.mkdir(exist_ok=True)

rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

pixel_columns = [f"pixel_{i}" for i in range(N_FEATURES)]
raw = {c: rng.uniform(0.1, 1.0, N_SAMPLES) for c in pixel_columns}
noise_alpha = rng.normal(0, 0.35, N_SAMPLES)
noise_beta = rng.laplace(0, 0.15, N_SAMPLES)
raw["class_raw"] = (
    np.sin(raw["pixel_0"]) * 3.5
    - np.cos(raw["pixel_32"]) * 2.8
    + np.log1p(raw["pixel_15"]) * 2.5 / (raw["pixel_45"] + 0.1)
    + noise_alpha * noise_beta
)

df = (
    pl.DataFrame(raw)
    .lazy()
    .with_columns(
        (pl.col("class_raw") > 3.0).cast(pl.Int64).alias("label"),
        pl.sum_horizontal(pixel_columns).alias("pixel_energy_sum"),
        pl.fold(
            acc=pl.lit(1.0),
            function=lambda a, b: a * b,
            exprs=[pl.col(c) for c in pixel_columns[:4]],
        ).alias("interaction_core"),
    )
    .drop("class_raw")
    .collect()
    .with_columns(
        (pl.int_range(pl.len()).over("label") % N_SPLITS).alias("fold")
    )
)

print(f"dados: {df.shape} | positivos: {df['label'].mean():.3f}")


def split(frame, folds):
    sub = frame.filter(pl.col("fold").is_in(folds))
    return sub.select(pixel_columns).to_numpy(), sub["label"].to_numpy()


cv_rows = []
for f in range(N_SPLITS):
    X_tr, y_tr = split(df, [i for i in range(N_SPLITS) if i != f])
    X_va, y_va = split(df, [f])
    knn = KNeighborsClassifier(n_neighbors=15, weights="distance", algorithm="brute", n_jobs=-1)
    dt = DecisionTreeClassifier(criterion="entropy", max_depth=12, min_samples_split=25, ccp_alpha=0.0005, random_state=SEED)
    cv_rows.append(
        {
            "fold": f,
            "knn": accuracy_score(y_va, knn.fit(X_tr, y_tr).predict(X_va)),
            "dt": accuracy_score(y_va, dt.fit(X_tr, y_tr).predict(X_va)),
        }
    )
cv = pl.DataFrame(cv_rows)
print(f"CV 10-fold acc | KNN {cv['knn'].mean():.4f} ± {cv['knn'].std():.4f} | DT {cv['dt'].mean():.4f} ± {cv['dt'].std():.4f}")

train_folds = [i for i in range(N_SPLITS) if i not in (TEST_FOLD, VAL_FOLD)]
X_train, y_train = split(df, train_folds)
X_val, y_val = split(df, [VAL_FOLD])
X_test, y_test = split(df, [TEST_FOLD])
df_test = df.filter(pl.col("fold") == TEST_FOLD)

grid = GridSearchCV(
    KNeighborsClassifier(algorithm="brute", n_jobs=-1),
    {"n_neighbors": [5, 11, 15], "weights": ["distance"], "metric": ["euclidean"]},
    cv=3,
    scoring="neg_log_loss",
    n_jobs=-1,
)
grid.fit(X_train[:3000], y_train[:3000])
knn_model = grid.best_estimator_.fit(X_train, y_train)
dt_model = DecisionTreeClassifier(
    criterion="entropy", max_depth=12, min_samples_split=25, ccp_alpha=0.0005, random_state=SEED
).fit(X_train, y_train)
print(f"KNN grid: {grid.best_params_}")

preds_knn = knn_model.predict_proba(X_test)[:, 1]
preds_dt = dt_model.predict_proba(X_test)[:, 1]


def to_tensor(X, y):
    xt = torch.tensor(X, dtype=torch.float32).view(-1, 1, 8, 8).to(device)
    yt = torch.tensor(y, dtype=torch.float32).view(-1, 1).to(device)
    return xt, yt


X_train_ts, y_train_ts = to_tensor(X_train, y_train)
X_val_ts, y_val_ts = to_tensor(X_val, y_val)
X_test_ts, _ = to_tensor(X_test, y_test)

train_loader = DataLoader(TensorDataset(X_train_ts, y_train_ts), batch_size=256, shuffle=True, drop_last=True)
val_loader = DataLoader(TensorDataset(X_val_ts, y_val_ts), batch_size=512)


class SpatialSelfAttention(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.query = nn.Conv2d(channels, channels // 4, 1)
        self.key = nn.Conv2d(channels, channels // 4, 1)
        self.value = nn.Conv2d(channels, channels, 1)
        self.gamma = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        b, c, h, w = x.shape
        q = self.query(x).view(b, -1, h * w).permute(0, 2, 1)
        k = self.key(x).view(b, -1, h * w)
        attn = torch.softmax(torch.bmm(q, k), dim=-1)
        v = self.value(x).view(b, -1, h * w)
        out = torch.bmm(v, attn.permute(0, 2, 1)).view(b, c, h, w)
        return self.gamma * out + x


class AttentionResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.BatchNorm2d(channels),
            nn.ReLU(),
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.BatchNorm2d(channels),
        )
        self.attention = SpatialSelfAttention(channels)

    def forward(self, x):
        return torch.relu(self.attention(self.conv(x)) + x)


class AttentionCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU())
        self.block = AttentionResidualBlock(32)
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d((2, 2)),
            nn.Flatten(),
            nn.Linear(32 * 4, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.head(self.block(self.stem(x)))


class HilbertSpaceQNN(nn.Module):
    def __init__(self):
        super().__init__()
        scale = 2.0 / np.sqrt(32)
        self.prep = nn.Linear(64, 32)
        self.theta = nn.Parameter(torch.randn(32, 32) * scale)
        self.phi = nn.Parameter(torch.randn(32, 32) * scale)
        self.readout = nn.Linear(32, 1)

    def forward(self, x):
        psi = torch.tanh(self.prep(x.flatten(1)))
        psi = psi @ torch.sin(self.theta)
        psi = psi @ torch.cos(self.phi)
        return self.readout(psi)


class EarlyStopper:
    def __init__(self, model, patience):
        self.patience = patience
        self.best = float("inf")
        self.wait = 0
        self.done = False
        self.state = copy.deepcopy(model.state_dict())

    def step(self, model, val_loss):
        if val_loss < self.best:
            self.best, self.wait = val_loss, 0
            self.state = copy.deepcopy(model.state_dict())
        else:
            self.wait += 1
            self.done = self.wait >= self.patience


criterion = nn.BCEWithLogitsLoss()


def train_epoch(model, optimizer):
    model.train()
    total = 0.0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        total += loss.item()
    return total / len(train_loader)


@torch.no_grad()
def evaluate_loss(model):
    model.eval()
    total = sum(nn.functional.binary_cross_entropy_with_logits(model(xb), yb, reduction="sum").item() for xb, yb in val_loader)
    return total / len(val_loader.dataset)


models = {"CNN + Attention": AttentionCNN().to(device), "QNN Multi-Phase": HilbertSpaceQNN().to(device)}
optimizers = {k: optim.AdamW(m.parameters(), lr=3e-3, weight_decay=1e-3) for k, m in models.items()}
schedulers = {k: optim.lr_scheduler.CosineAnnealingLR(o, T_max=N_EPOCHS, eta_min=1e-6) for k, o in optimizers.items()}
stoppers = {k: EarlyStopper(m, PATIENCE) for k, m in models.items()}

history, weight_stream = [], []

for epoch in range(1, N_EPOCHS + 1):
    for name, model in models.items():
        if stoppers[name].done:
            continue
        loss = train_epoch(model, optimizers[name])
        val_loss = evaluate_loss(model)
        schedulers[name].step()
        stoppers[name].step(model, val_loss)
        history.append({"epoch": epoch, "model": name, "loss": loss, "val_loss": val_loss})
        if name == "QNN Multi-Phase":
            weight_stream.append({"epoch": epoch, "parameter": "theta_norm", "value": model.theta.norm().item()})
            weight_stream.append({"epoch": epoch, "parameter": "phi_norm", "value": model.phi.norm().item()})
        if stoppers[name].done:
            print(f"early stopping: {name} na época {epoch} (melhor val_loss {stoppers[name].best:.4f})")
    if all(s.done for s in stoppers.values()):
        break

for name, model in models.items():
    model.load_state_dict(stoppers[name].state)
    model.eval()


@torch.no_grad()
def predict_proba(model, X):
    return torch.sigmoid(model(X)).cpu().numpy().ravel()


preds_cnn = predict_proba(models["CNN + Attention"], X_test_ts)
preds_qnn = predict_proba(models["QNN Multi-Phase"], X_test_ts)

all_preds = {"KNN": preds_knn, "Decision Tree": preds_dt, "CNN + Attention": preds_cnn, "QNN Multi-Phase": preds_qnn}
metrics = pl.DataFrame(
    [
        {
            "model": name,
            "accuracy": accuracy_score(y_test, p > 0.5),
            "f1": f1_score(y_test, p > 0.5),
            "roc_auc": roc_auc_score(y_test, p),
            "log_loss": log_loss(y_test, np.clip(p, 1e-7, 1 - 1e-7)),
        }
        for name, p in all_preds.items()
    ]
)
print(f"\nmétricas no fold de teste (n={len(y_test)})")
print(metrics.with_columns(pl.col(pl.Float64).round(4)))


def gradient_shap(model, x, background, n_draws=64):
    model.eval()
    attr = torch.zeros_like(x)
    for _ in range(n_draws):
        base = background[torch.randint(len(background), (len(x),), device=x.device)]
        alpha = torch.rand(len(x), 1, 1, 1, device=x.device)
        point = (base + alpha * (x - base)).requires_grad_(True)
        (grad,) = torch.autograd.grad(model(point).sum(), point)
        attr += grad * (x - base)
    return attr / n_draws


shap_values = gradient_shap(models["CNN + Attention"], X_test_ts[:200], X_train_ts[:500])
shap_importance = shap_values.abs().mean(dim=0).view(-1).cpu().numpy()
df_shap = pl.DataFrame({"feature": pixel_columns, "shap_importance": shap_importance}).sort("shap_importance", descending=True)
print("\ntop 5 SHAP:", df_shap.head(5)["feature"].to_list())

df_results = df_test.with_columns(
    pl.Series("pred_knn", preds_knn),
    pl.Series("pred_dt", preds_dt),
    pl.Series("pred_cnn", preds_cnn),
    pl.Series("pred_qnn", preds_qnn),
)
df_results.write_parquet(OUT / "pipeline_output.parquet", compression="snappy")

df_history = pl.DataFrame(history)
df_curves = df_history.unpivot(index=["epoch", "model"], on=["loss", "val_loss"], variable_name="split", value_name="loss_value")
alt.Chart(df_curves.to_pandas()).mark_line(point=True).encode(
    x=alt.X("epoch:O", title="Época"),
    y=alt.Y("loss_value:Q", title="BCE", scale=alt.Scale(zero=False)),
    color="model:N",
    strokeDash="split:N",
).properties(title="Curvas de treino e validação", width=650, height=240).save(str(OUT / "curvas.html"))

px.line(
    pl.DataFrame(weight_stream).to_pandas(),
    x="epoch",
    y="value",
    color="parameter",
    title="Norma L2 dos parâmetros do ansatz (QNN)",
).update_layout(plot_bgcolor="white", width=750, height=330).write_html(OUT / "qnn_norms.html")

names = {"pred_knn": "KNN", "pred_dt": "Decision Tree", "pred_cnn": "CNN + Attention", "pred_qnn": "QNN Multi-Phase"}
df_melt = (
    df_results.select("label", *names)
    .unpivot(index="label", on=list(names), variable_name="model_key", value_name="probability")
    .with_columns(
        pl.col("label").cast(pl.Utf8).alias("classe"),
        pl.col("model_key").replace(names).alias("modelo"),
    )
    .to_pandas()
)
(
    ggplot(df_melt, aes(x="probability", fill="classe"))
    + geom_density(alpha=0.45)
    + facet_wrap("~modelo", nrow=2)
    + theme_minimal()
    + labs(title="Distribuição das probabilidades no fold de teste", x="Probabilidade prevista", y="Densidade")
    + theme(figure_size=(9, 5))
).save(OUT / "densidades.png", dpi=130, verbose=False)

fig, ax = plt.subplots(figsize=(9, 4))
(
    so.Plot(df_shap.head(10).to_pandas(), x="shap_importance", y="feature")
    .add(so.Bar(color="#2563eb"))
    .label(title="Top 10 pixels por Gradient SHAP (CNN)", x="|SHAP| médio", y="")
    .on(ax)
    .plot()
)
fig.tight_layout()
fig.savefig(OUT / "shap_top10.png", dpi=130)
plt.close(fig)

print(f"\nartefatos em {OUT.resolve()}")

Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
Device type: tpu, Device index: default
Initializing TPU distributed runtime


dados: (30000, 68) | positivos: 0.150
CV 10-fold acc | KNN 0.8583 ± 0.0012 | DT 0.9704 ± 0.0029
KNN grid: {'metric': 'euclidean', 'n_neighbors': 15, 'weights': 'distance'}
early stopping: QNN Multi-Phase na época 9 (melhor val_loss 0.0582)
early stopping: CNN + Attention na época 10 (melhor val_loss 0.0689)

métricas no fold de teste (n=3001)
shape: (4, 5)
┌─────────────────┬──────────┬────────┬─────────┬──────────┐
│ model           ┆ accuracy ┆ f1     ┆ roc_auc ┆ log_loss │
│ ---             ┆ ---      ┆ ---    ┆ ---     ┆ ---      │
│ str             ┆ f64      ┆ f64    ┆ f64     ┆ f64      │
╞═════════════════╪══════════╪════════╪═════════╪══════════╡
│ KNN             ┆ 0.8567   ┆ 0.1004 ┆ 0.8769  ┆ 0.3103   │
│ Decision Tree   ┆ 0.9713   ┆ 0.9036 ┆ 0.9836  ┆ 0.1323   │
│ CNN + Attention ┆ 0.9723   ┆ 0.9105 ┆ 0.9961  ┆ 0.0604   │
│ QNN Multi-Phase ┆ 0.9767   ┆ 0.9222 ┆ 0.9967  ┆ 0.0538   │
└─────────────────┴──────────┴────────┴─────────┴──────────┘

top 5 SHAP: ['pixel_45', 'pixe